## **Ensemble Methods**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingRegressor, RandomForestRegressor, AdaBoostRegressor,
    GradientBoostingRegressor, VotingRegressor, StackingRegressor
)
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

df = pd.read_csv('china_used_cars.csv')
drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name:20s} MAE: {mean_absolute_error(y_test, pred):10.1f}  R2: {r2_score(y_test, pred):.4f}")

evaluate("Decision Tree", DecisionTreeRegressor(random_state=42))
evaluate("Bagging", BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), n_estimators=200, random_state=42, n_jobs=-1))
evaluate("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
evaluate("AdaBoost", AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4, random_state=42), n_estimators=200, learning_rate=0.5, random_state=42))
evaluate("Gradient Boosting", GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42))
evaluate("XGBoost", xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1))

base = [
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LinearRegression())),
]
evaluate("Voting", VotingRegressor(estimators=base, weights=[2, 2, 1]))
evaluate("Stacking", StackingRegressor(estimators=base, final_estimator=RidgeCV(), cv=5, n_jobs=-1))

Decision Tree        MAE:    12486.2  R2: 0.5948
Bagging              MAE:    12027.3  R2: 0.7401
Random Forest        MAE:    12357.6  R2: 0.7044
AdaBoost             MAE:    40258.0  R2: 0.5645
Gradient Boosting    MAE:    18772.0  R2: 0.4449
XGBoost              MAE:    15825.6  R2: 0.5394
Voting               MAE:    17487.1  R2: 0.5978
Stacking             MAE:    17187.3  R2: 0.7057
